# Day 5: Building a Chatbot

Yesterday we learned the most important fact about chat models: they remember nothing. A chatbot only *seems* to remember because we keep a list of every message and send the whole list each time.

Today LangGraph takes over that job. We'll use two things:

- **`add_messages`**: a reducer (remember Day 2?) that knows how to handle chat messages
- **`MessagesState`**: a ready-made state with a `messages` key that already uses that reducer

Then we'll build a real chatbot, give it a personality, keep its history from growing forever, and finish with a small study-buddy bot that changes behaviour depending on what you ask.

In [ ]:
%pip install -q -U langgraph langchain-openai python-dotenv

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI
from IPython.display import Image, display

In [ ]:
load_dotenv(find_dotenv(usecwd=True))
assert os.getenv("OPENAI_API_KEY"), "Add OPENAI_API_KEY to the .env file in the repo root (see Day 4)."
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-4.1-nano"), temperature=0)

In [ ]:
def show_graph(app):
    try:
        display(Image(app.get_graph().draw_mermaid_png()))
    except Exception:
        print(app.get_graph().draw_mermaid())

## 1. `add_messages`: a reducer made for chat

On Day 2 we used `operator.add` to make a list grow. You could use that for messages too, but chat needs a few extra tricks. Think of `add_messages` as a smart notebook for a conversation:

1. It **appends** new messages to the end.
2. It **converts** shortcuts like `("user", "hi")` into proper message objects.
3. It gives every message an **id**, like a page number.
4. If a new message has the **same id** as an old one, it **replaces** it (like editing a WhatsApp message).
5. If you send a `RemoveMessage(id=...)`, it **deletes** that message.

The nice part is we can call it directly, outside any graph, to see each rule in action.

In [ ]:
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, RemoveMessage

In [ ]:
# Rules 1, 2 and 3: append, convert, give ids
messages = add_messages([], [("user", "Hello!"), ("assistant", "Hi there!")])

for m in messages:
    print(f"{m.type:5} | {m.content:10} | id starts with {m.id[:8]}")

### Predict before you run

We'll send a new `HumanMessage` that reuses the **id** of the first message, with the text "Hello (edited)!". How many messages will the list have afterwards: 2 or 3?

<details>
<summary>Click to see the answer</summary>

Still 2. Because the id matches, `add_messages` treats it as an edit and replaces the old message instead of adding a new one.
</details>

In [ ]:
first_id = messages[0].id

# Rule 4: same id means replace
messages = add_messages(messages, [HumanMessage(content="Hello (edited)!", id=first_id)])
print(len(messages), [m.content for m in messages])

# Rule 5: RemoveMessage deletes by id
messages = add_messages(messages, [RemoveMessage(id=first_id)])
print(len(messages), [m.content for m in messages])

## 2. `MessagesState`: the ready-made chat state

You could write the chat state yourself, like this:

```python
class MyChatState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
```

But you'd be writing the same thing in every project, so LangGraph ships it already done as `MessagesState`. If you need more keys, just inherit from it and add them.

In [ ]:
from langgraph.graph import StateGraph, START, END, MessagesState

In [ ]:
class ChatState(MessagesState):
    user_name: str          # an extra key, alongside the built-in "messages"

In [ ]:
print(list(ChatState.__annotations__))

## 3. Your first chatbot graph

```
  START --> chatbot --> END
```

The node does three things. It takes all the messages so far, sends them to the model, and returns **only the new reply**. The reducer takes care of adding it to the list.

In [ ]:
def chatbot(state: MessagesState) -> dict:
    reply = llm.invoke(state["messages"])     # the model sees the whole conversation
    return {"messages": [reply]}              # but we return only the new message, inside a list

In [ ]:
builder = StateGraph(MessagesState)

In [ ]:
builder.add_node("chatbot", chatbot)

In [ ]:
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

In [ ]:
chat_app = builder.compile()
show_graph(chat_app)

### Predict before you run

We send one user message. How many messages will be in the result?

<details>
<summary>Click to see the answer</summary>

Two: your message, plus the model's reply. The node returned one message and `add_messages` appended it to the one we sent in.
</details>

In [ ]:
result = chat_app.invoke({"messages": [("user", "Give me one fun fact about India.")]})

for m in result["messages"]:
    m.pretty_print()

## 4. Giving the bot a personality

You'll want your bot to behave a certain way: friendly, short answers, a particular tone. That's what the system message is for.

Here's a habit I recommend: **add the system message inside the node, right before calling the model. Don't store it in the state.** Two reasons:

- the saved conversation stays clean, it only holds what the user and bot actually said
- you can change the bot's instructions later without rewriting every old conversation

It's like a call-centre script on the agent's desk. The customer never sees it, and the company can update it any time.

In [ ]:
PERSONA = SystemMessage(content=(
    "You are Chai, a friendly Python tutor for beginners. "
    "Keep answers under 60 words and always include a tiny code example."
))

In [ ]:
def tutor(state: MessagesState) -> dict:
    reply = llm.invoke([PERSONA] + state["messages"])    # persona added here, not stored
    return {"messages": [reply]}

In [ ]:
builder = StateGraph(MessagesState)

In [ ]:
builder.add_node("tutor", tutor)

In [ ]:
builder.add_edge(START, "tutor")
builder.add_edge("tutor", END)

In [ ]:
tutor_app = builder.compile()

In [ ]:
result = tutor_app.invoke({"messages": [("user", "What is a list comprehension?")]})
print(result["messages"][-1].content)

In [ ]:
# The system message was used, but it isn't saved in the conversation
print([m.type for m in result["messages"]])

### Your turn

Change `PERSONA` to something completely different, maybe a bot that only answers in Hinglish, or one that always ends with a follow-up question. Rebuild and ask the same question.

In [ ]:
PERSONA = SystemMessage(content="You are a Python tutor who explains in simple Hinglish. Max 50 words.")  # change this

In [ ]:
builder = StateGraph(MessagesState)

In [ ]:
builder.add_node("tutor", tutor)

In [ ]:
builder.add_edge(START, "tutor")
builder.add_edge("tutor", END)

In [ ]:
tutor_app = builder.compile()

In [ ]:
print(tutor_app.invoke({"messages": [("user", "What is a list comprehension?")]})["messages"][-1].content)

## 5. Having a real back-and-forth

Each `invoke()` starts from a blank slate. So for now, we carry the history ourselves: take the messages that came back and pass them in again, with the new message added. (On Day 7, a *checkpointer* will do this automatically and you'll never write this loop again. But it's worth doing by hand once so you know what's happening underneath.)

In [ ]:
history = []

for user_text in ["Hi, I'm Rohan and I love cricket.", "What's my name and what do I like?"]:
    result = tutor_app.invoke({"messages": history + [HumanMessage(content=user_text)]})
    history = result["messages"]          # keep everything for the next turn
    print("You:", user_text)
    print("Bot:", history[-1].content)
    print()

print("Messages in history now:", len(history))

If you'd like to actually chat with it, uncomment the last line in the cell below and run it. Type `quit` to stop.

In [ ]:
def chat():
    history = []
    while True:
        text = input("You: ")
        if text.lower() in ("quit", "exit", "q"):
            break
        history = tutor_app.invoke({"messages": history + [("user", text)]})["messages"]
        print("Bot:", history[-1].content)

# chat()

## 6. Stopping the history from growing forever

Every message you keep is sent to the model on every turn. A 200-message conversation means paying for 200 messages, every time the user types "ok". It also gets slower, and eventually hits the model's limit.

A simple fix is to only send the most recent messages. LangChain has a helper for this called `trim_messages`. Think of it as only reading the last page of a long chat, not the whole history.

In [ ]:
from langchain_core.messages import trim_messages

In [ ]:
long_chat = add_messages([], [
    ("user", "msg 1"), ("assistant", "reply 1"),
    ("user", "msg 2"), ("assistant", "reply 2"),
    ("user", "msg 3"), ("assistant", "reply 3"),
    ("user", "msg 4"),
])

recent = trim_messages(
    long_chat,
    strategy="last",       # keep the newest messages
    token_counter=len,     # count messages instead of tokens, simpler to reason about
    max_tokens=3,          # so this means "keep at most 3 messages"
    start_on="human",      # the kept part should start with a user message
)
print([m.content for m in recent])

Use it inside the node. The model only sees the recent part, while the full history is still kept in the state.

In [ ]:
def short_memory_bot(state: MessagesState) -> dict:
    recent = trim_messages(state["messages"], strategy="last", token_counter=len,
                           max_tokens=6, start_on="human")
    return {"messages": [llm.invoke([PERSONA] + recent)]}

In [ ]:
builder = StateGraph(MessagesState)

In [ ]:
builder.add_node("bot", short_memory_bot)

In [ ]:
builder.add_edge(START, "bot")
builder.add_edge("bot", END)

In [ ]:
short_bot = builder.compile()

In [ ]:
print(short_bot.invoke({"messages": long_chat})["messages"][-1].content)

## 7. Common mistakes

**Mistake 1: Returning the whole history from the node.**
This is by far the most common chatbot bug I see. It looks harmless:

```python
return {"messages": state["messages"] + [reply]}
```

But `add_messages` *appends* whatever you return. You're handing it the full history again, so... does everything get duplicated? Let's check.

In [ ]:
def returns_everything(state: MessagesState) -> dict:
    reply = llm.invoke(state["messages"])
    return {"messages": state["messages"] + [reply]}

In [ ]:
builder = StateGraph(MessagesState)

In [ ]:
builder.add_node("bot", returns_everything)

In [ ]:
builder.add_edge(START, "bot")
builder.add_edge("bot", END)

In [ ]:
app = builder.compile()

In [ ]:
result = app.invoke({"messages": [("user", "hello")]})
print(len(result["messages"]), "messages")

Interesting. You get 2, not 3. Because the old messages carry the same ids, `add_messages` treats them as "replace with the same thing" rather than new messages. So you got lucky here. But with `operator.add` (Day 2) this exact code would duplicate everything, and it's wasteful even here. **Return only the new message.** It's clearer and it's always correct.

**Mistake 2: Forgetting the list.**
`return {"messages": reply}` instead of `return {"messages": [reply]}`. It happens to work for a single message, but returning a list is the habit that never surprises you, especially once a node returns two or three messages at once.

**Mistake 3: Storing the system prompt in the state.**
If you put the `SystemMessage` into `messages`, it's saved with the conversation. Later you change your prompt, and old conversations still carry the old one, or worse, two system messages at once. Add it inside the node instead.

**Mistake 4: `trim_messages` cutting the conversation in a weird place.**
Without `start_on="human"`, the trimmed history might start with an AI reply, which confuses some models. Keep `start_on="human"`.

## 8. Mini project: Study Buddy

Let's combine Day 3 (routing) with today's chatbot. The Study Buddy looks at your message and decides what kind of help you want:

- "quiz me on..." gets a multiple-choice question
- "summarize..." gets three bullet points
- anything else gets a simple explanation

```
                       +--> explain --+
  START --> detect_mode +--> quiz ----+--> END
                       +--> summary --+
```

In [ ]:
from typing import Literal

In [ ]:
class StudyState(MessagesState):
    mode: str

In [ ]:
INSTRUCTIONS = {
    "explain": "You are a patient teacher. Explain simply with one example. Max 80 words.",
    "quiz":    "Ask ONE multiple-choice question (A to D) about the topic. Do not reveal the answer.",
    "summary": "Summarize the topic in exactly 3 short bullet points.",
}

In [ ]:
def detect_mode(state: StudyState) -> dict:
    # A simple keyword check. On Day 10 we'll let the LLM make this decision instead.
    text = state["messages"][-1].content.lower()
    if "quiz" in text or "test me" in text:
        return {"mode": "quiz"}
    if "summar" in text:
        return {"mode": "summary"}
    return {"mode": "explain"}

In [ ]:
def make_helper(mode):
    # One function that builds a node for each mode, so we don't repeat ourselves
    def helper(state: StudyState) -> dict:
        reply = llm.invoke([SystemMessage(content=INSTRUCTIONS[mode])] + state["messages"])
        return {"messages": [reply]}
    return helper

In [ ]:
def route_mode(state: StudyState) -> Literal["explain", "quiz", "summary"]:
    return state["mode"]

In [ ]:
builder = StateGraph(StudyState)

In [ ]:
builder.add_node("detect_mode", detect_mode)

In [ ]:
for mode in INSTRUCTIONS:
    builder.add_node(mode, make_helper(mode))
    builder.add_edge(mode, END)

In [ ]:
builder.add_edge(START, "detect_mode")
builder.add_conditional_edges("detect_mode", route_mode)

In [ ]:
study = builder.compile()
show_graph(study)

In [ ]:
for question in ["What is recursion?", "Quiz me on Python dictionaries", "Summarize what LangGraph is"]:
    result = study.invoke({"messages": [("user", question)]})
    print("You :", question)
    print("Mode:", result["mode"])
    print("Bot :", result["messages"][-1].content)
    print()

## 9. Practice

**Exercise 1.** Make a chatbot whose state also has `turns: int`. Every time the bot replies, `turns` goes up by 1. Have a three-message conversation (passing the history and turns back in each time) and check that `turns` ends at 3.

**Exercise 2.** Write a node called `forget_old` that deletes everything except the last 2 messages using `RemoveMessage`. Put it **after** the chatbot node.

In [ ]:
# Solution 1
class CountingState(MessagesState):
    turns: int

In [ ]:
def counting_bot(state: CountingState) -> dict:
    reply = llm.invoke(state["messages"])
    return {"messages": [reply], "turns": state.get("turns", 0) + 1}

In [ ]:
builder = StateGraph(CountingState)

In [ ]:
builder.add_node("bot", counting_bot)

In [ ]:
builder.add_edge(START, "bot")
builder.add_edge("bot", END)

In [ ]:
counting_app = builder.compile()

In [ ]:
state = {"messages": [], "turns": 0}
for text in ["hi", "how are you?", "bye"]:
    state = counting_app.invoke({"messages": state["messages"] + [("user", text)],
                                 "turns": state["turns"]})

print("turns:", state["turns"], "| messages:", len(state["messages"]))
assert state["turns"] == 3 and len(state["messages"]) == 6

In [ ]:
# Solution 2
def forget_old(state: MessagesState) -> dict:
    # Every message except the last two gets a RemoveMessage
    return {"messages": [RemoveMessage(id=m.id) for m in state["messages"][:-2]]}

In [ ]:
builder = StateGraph(MessagesState)

In [ ]:
builder.add_node("chatbot", chatbot)
builder.add_node("forget_old", forget_old)

In [ ]:
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", "forget_old")
builder.add_edge("forget_old", END)

In [ ]:
app = builder.compile()

In [ ]:
result = app.invoke({"messages": [("user", "a"), ("assistant", "b"), ("user", "c")]})
print([m.type for m in result["messages"]])
assert len(result["messages"]) == 2

## Wrapping up

`add_messages` is the reducer behind every chatbot: it appends, converts shortcuts, assigns ids, replaces and removes. `MessagesState` is that reducer, packaged and ready to use, and you can extend it with your own keys.

Your chat node should return **only the new reply**, in a list. Add the system prompt inside the node, and use `trim_messages` so long chats don't get slow and expensive.

So far our bot can only talk. Tomorrow we give it **tools**, things it can actually *do*, like calculate, look up data or call an API. That's the step from chatbot to agent.